### ML_farm_weather.csv → 기상 피처 4개 생성 → gold.farm_weather_features
##### (ML 학습용 선별 농장 × 날씨 피처)
##### grain: farm_id + reference date
##### 6개월(180d) / 1년(365d)


7일치 한번에?

In [ ]:
import sys
sys.path.append("/Workspace/방역로/00_Shared_Utils")
from utils_config import CATALOG
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
# 최소 유효일수 기준 — 팀원과 협의 후 확정
# MIN_VALID_DAYS = 5

In [ ]:
df = (
    spark.table(f"{CATALOG}.silver.farm_weather")
    .withColumn("reference_date", F.to_date(F.col("reference_date"), "yyyy-MM-dd"))
    .withColumn("TM", F.to_date(F.col("TM"), "yyyy-MM-dd"))
    # 이상치 처리
    .withColumn("HM_AVG",
        F.when((F.col("HM_AVG") < 0) | (F.col("HM_AVG") > 100), None)
         .otherwise(F.col("HM_AVG")))
    .withColumn("RN_DAY",
        F.when(F.col("RN_DAY") < 0, None).otherwise(F.col("RN_DAY")))
    .withColumn("WS_AVG",
        F.when(F.col("WS_AVG") < 0, None).otherwise(F.col("WS_AVG")))
)

In [ ]:
# humidity: TM == reference_date 당일 HM_AVG
humidity = (
    df.filter(F.col("TM") == F.col("reference_date"))
      .select("farm_id", "reference_date",
              F.col("HM_AVG").alias("humidity"))
)

In [ ]:
# 7일 집계: [reference_date-7, reference_date) — reference_date 당일 미포함
df_7d = df.filter(
    (F.col("TM") >= F.date_sub(F.col("reference_date"), 7)) &
    (F.col("TM") <  F.col("reference_date"))
)

weather_7d = (
    df_7d.groupBy("farm_id", "reference_date").agg(
        # 기온 이상치 기준은 팀원과 협의 후 추가 예정
        F.min("TA_MIN").alias("min_temp_7d"),

        # RN_DAY 전체 NULL → NULL 유지, 나머지 NULL → 0(무강수)
        F.when(
            F.count(F.col("RN_DAY")) == 0, None
        ).otherwise(
            F.sum(F.coalesce(F.col("RN_DAY"), F.lit(0.0)))
        ).alias("precipitation_7d"),

        # 음수는 이미 위에서 NULL 처리됨
        F.avg("WS_AVG").alias("wind_speed_avg_7d"),

        # 유효일수 — MIN_VALID_DAYS 확정 후 아래 주석 해제
        # F.count("TA_MIN").alias("valid_days_temp"),
        # F.count("RN_DAY").alias("valid_days_rain"),
        # F.count("WS_AVG").alias("valid_days_wind"),
    )
    # 최소 유효일수 미충족 시 NULL 처리 — MIN_VALID_DAYS 확정 후 아래 주석 해제
    # .withColumn("min_temp_7d",
    #     F.when(F.col("valid_days_temp") < MIN_VALID_DAYS, None)
    #      .otherwise(F.col("min_temp_7d")))
    # .withColumn("wind_speed_avg_7d",
    #     F.when(F.col("valid_days_wind") < MIN_VALID_DAYS, None)
    #      .otherwise(F.col("wind_speed_avg_7d")))
    # .withColumn("precipitation_7d",
    #     F.when(F.col("valid_days_rain") < MIN_VALID_DAYS, None)
    #      .otherwise(F.col("precipitation_7d")))
)

In [ ]:
result = humidity.join(weather_7d, ["farm_id", "reference_date"], "left")

In [ ]:

farm_ids = (
    spark.table(f"{CATALOG}.silver.farm_master")
    .select("farm_id").distinct()
)

(
    result.join(farm_ids, "farm_id", "inner")
          .writeTo(f"{CATALOG}.gold.farm_weather_features_daily")
          .using("delta")
          .createOrReplace()
)
print("저장 완료: farm_weather_features_daily")

In [ ]:
%sql
select * from dt4_team1_databricks.gold.farm_weather_features_daily limit(5)